In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader 
import torchvision
from torchvision.datasets import MNIST
from torchvision.transforms import transforms
import torch.optim as optim

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

train_dataset = MNIST(root="./data",train=True,transform=transform,download=True)
test_dataset = MNIST(root="./data",train=False,transform=transform,download=True)

In [3]:
trainloader = DataLoader(train_dataset,batch_size=64,shuffle=True)
testloader = DataLoader(test_dataset,batch_size=64)

### CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1,28,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(28,56,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(7*7*56,112),
            nn.ReLU(),

            nn.Linear(112,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) #flattening
        x = self.fc_layers(x)

        return x
        

In [5]:
model = CNN()

creterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [6]:
epochs = 10
epoch_training_loss = 0.0
for epoch in range(epochs):
    training_loss = 0.0 
    for images,labels in trainloader:
        
        model.train()

        optimizer.zero_grad()
        outputs = model.forward(images)
        loss = creterion(outputs,labels)
        loss.backward()
        optimizer.step()

        training_loss +=loss.item()
    epoch_training_loss = training_loss / len(trainloader)

    print(f"{epoch+1} and trainig loss is {epoch_training_loss}")
    
        
        

1 and trainig loss is 0.16389756857542626
2 and trainig loss is 0.04767194433822366
3 and trainig loss is 0.032631063390557824
4 and trainig loss is 0.024031392550489344
5 and trainig loss is 0.01914894583481508
6 and trainig loss is 0.014728237897719765
7 and trainig loss is 0.01172315226477546
8 and trainig loss is 0.009547760481933992
9 and trainig loss is 0.006528038083377887
10 and trainig loss is 0.007907294852962715


In [7]:
correct = 0
total = 0
with torch.no_grad():
    for images,labels in testloader:
        model.eval()

        outputs = model.forward(images)

        _,predicted = torch.max(outputs,1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
print(f"accuracy = {correct/total*100}")

    

accuracy = 99.05000000000001


### RNN 

In [24]:
class RNN(nn.Module):
    def __init__(self,input_size=28,hidden_size=128,num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.RNN = nn.RNN(input_size,hidden_size=128,num_layers=1,batch_first = True)

        self.fc = nn.Linear(hidden_size , 10)
        
    def forward(self,x):
        h0 =  torch.zeros(self.num_layers, x.size(0),self.hidden_size)
        out,_ = self.RNN(x,h0)

        out =  self.fc(out[:,-1,:])
        return out

In [25]:
model = RNN()
creterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [26]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for images,labels in trainloader:
        optimizer.zero_grad()
        images = images.squeeze(1)
        outputs = model(images) # (batch_size,1)


        loss = creterion(outputs,labels)
        loss.backward()
        optimizer.step()

    print(f"{epoch+1} and loss is {loss.item()}")


1 and loss is 0.30310603976249695
2 and loss is 0.1417902559041977
3 and loss is 0.1103631928563118
4 and loss is 0.13880802690982819
5 and loss is 0.2827346622943878
6 and loss is 0.13576695322990417
7 and loss is 0.10414721816778183
8 and loss is 0.2725072503089905
9 and loss is 0.1451982706785202
10 and loss is 0.18555526435375214


In [29]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images,labels in testloader:
        images = images.squeeze()
        outputs = model(images)
        _,predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"accuracy = {correct/total*100}")

accuracy = 96.64
